# nb04: No-JEPA raw-TimesFM ceiling probe (DIAGNOSTIC ONLY)

* * *

**QUESTION:** Do RAW pretrained-backbone (TimesFM) features linearly decode our three downstream targets (future return, future volatility, return direction) on val — with NO JEPA, NO projection head, NO training — better than (a) the untrained random-encoder floor and (b) hand-crafted raw-input features? This measures the CEILING any frozen-backbone JEPA could inherit, before any training units are spent on a TimesFM pivot.

**H1:** RAW TimesFM pooled features decode future volatility materially better than both the untrained-random `z_price` floor and the raw-input baseline (signal exists in the pretrained representation; the project's failure is the loop/head, and a backbone swap is worth training).

**H0 (null):** RAW TimesFM features decode no better than a random projection (`≈` untrained floor). A pretrained backbone alone is not the bottleneck; the limitation is elsewhere (targets, conditioning, or the data itself).

**MEASURED TARGETS** (identical harness/seed/grid for all four representations):
- future_return: ridge val R² (and train R² for overfit watch) at best alpha in the WIDE grid.
- future_volatility: ridge val R² (and train R²) — the clearest target; the headline for the decision rule.
- direction = sign(future_return): logistic val accuracy + AUROC vs the majority baseline.
- Four feature extractors, all via the same `probe_*_features` harness: (1) raw-input stats, (2) untrained random `z_price` (the Stage 1 FLOOR, re-run in-session under the wide grid), (3) trained nb03_best `z_price`, (4) RAW TimesFM pooled hidden states (the CEILING).

**DECISION RULE.** Let `C` = TimesFM future_volatility val R², `Fl` = untrained-random val R², `Rw` = raw-input val R².
- **COMMIT** to the TimesFM pivot (next step: build a JEPA loop on a frozen TimesFM backbone, benchmarked against this floor) if `C > Fl + 0.03` AND `C > Rw + 0.02` — TimesFM materially beats both the random-projection floor and raw inputs.
- **HOLD** if `C ≈ Fl` (within ~0.01): TimesFM is no richer than a random projection for these targets → a backbone swap alone won't help; reconsider targets/conditioning before spending training units.
- If best direction accuracy across ALL four encoders `≈` the majority baseline (~0.53): H=16 direction is near-unpredictable from price alone → drop direction as a primary target; focus vol/return or add conditioning (options/futures/text).

**PRIORS / ASSUMPTIONS:**
- Inputs to every encoder are per-sample-normalized close log-returns (the collector re-normalizes context exactly as `PriceWindowDataset(normalize=True)`); price LEVEL never enters. Probe targets are computed from RAW (unnormalized) target windows so their scale is comparable across samples.
- Stage 1 pinned `best_alpha=100` (its grid max) on several targets → it was under-regularized and its val R² is slightly understated. So the floor/baseline here are RE-RUN under the wide grid `(0.01 … 1e4)`, NOT the recorded Stage 1 figures. TimesFM's higher feature dim (1280 ≫ 256) wants more regularization, hence the wide grid.
- TimesFM via the HuggingFace `transformers` integration (`TimesFmModelForPrediction`), NOT the `timesfm` pip package → stays on Colab's numpy-2 stack (needs `transformers>=4.48`). We tap `last_hidden_state` (mean-pooled over patches), not the forecast head.
- This is a MEASUREMENT, not a method change. JEPA remains the project; we are de-risking the pivot.

**FALSIFIER:** If RAW TimesFM future_volatility val R² is within ~0.01 of the untrained-random floor (and not above raw inputs by the stated margins), H1 is false: the pretrained backbone carries no more linearly-decodable volatility signal than a random projection, and a backbone swap alone will not rescue the pipeline.

**RESULT** (Colab, 2026-06-06; n_train=100 batches, n_val=50, seed=42, wide grid 0.01..1e4): future_volatility val R² — raw-input **+0.333**, untrained z_price (FLOOR) +0.097, trained z_price +0.118, **TimesFM (CEILING) −0.134**. future_return val R² — raw +0.012, untrained −0.041, trained −0.058, TimesFM −0.081 (all ≈ 0 / negative). direction val acc — raw 0.528, untrained **0.571** (AUROC 0.569), trained 0.535, TimesFM 0.539; majority 0.528, 2·SE=0.019. TimesFM overfits hard (vol train +0.246 vs val −0.134) and pins α at the grid max for vol/return and the grid min for direction. raw-input is NOT boundary-pinned on vol (α=1000).

**DECISION + NEXT ACTION:** H1 **REFUTED** / **HOLD on the TimesFM pivot.** C=−0.134 is *below* Fl=+0.097 (not above by +0.03) and far below Rw=+0.333 — raw close-channel TimesFM features are anti-informative for volatility on val, so a frozen-TimesFM JEPA loop is NOT justified on this evidence. The decisive surprise is that a trivial 4-moment baseline (+0.333) beats every learned/pretrained representation: the volatility signal is real and strong but trivially linearly accessible, and the per-sample context normalization nearly removes the very scale/volatility-level feature that predicts future volatility — the encoders (which see the same normalized context) fail to recover it while a crude moment probe does. NEXT (forward-passes only, no training units): (1) confirm what carries raw-input's +0.333 (probe on UN-normalized context, and a single feature = raw context std) to test the normalization-removes-scale hypothesis; (2) only if that’s inconclusive, run TimesFM variant='concat6' (all 6 channels) to close the channel confound. Do NOT spend training units on a backbone swap. Re-bar the project: any learned model must beat +0.33 vol R², not the +0.097 floor.


In [ ]:
# == Colab setup (skipped in VS Code / local kernel) ==
import os, sys

IN_COLAB = 'google.colab' in sys.modules
IN_VSCODE = 'VSCODE_PID' in os.environ or 'VSCODE_CWD' in os.environ
if IN_VSCODE:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content
    !git clone https://github.com/shreyasnat2804/JEPA-quant.git 2>/dev/null || (cd JEPA-quant && git pull)
    %cd /content/JEPA-quant
    # transformers>=4.48 is REQUIRED: the TimesFM integration (TimesFmModelForPrediction)
    # landed in 4.48. We use the HF integration, NOT the `timesfm` pip package — the pip
    # package can pin numpy<2 and break Colab's numpy-2 stack (same trap as Moirai/uni2ts).
    %pip install -q "transformers>=4.48" "peft>=0.11" accelerate einops matplotlib pyarrow
    # Colab preinstalls torchao 0.10.0. Recent PEFT raises (not returns False) when it
    # finds torchao below its 0.16.0 minimum, blowing up get_peft_model() during LoRA
    # dispatch even for plain LoRA. Remove it; upgrading would drag torch/ABI churn.
    %pip uninstall -q -y torchao

# Add src to path regardless of environment
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..' if 'notebooks' in os.getcwd() else '.'))
src_path = os.path.join(repo_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print('repo_root:', repo_root)
print('IN_COLAB:', IN_COLAB, '  IN_VSCODE:', IN_VSCODE)


In [ ]:
# == autoreload shim (imp removed in Python 3.12) ==
import types, importlib
if 'imp' not in sys.modules:
    _imp_shim = types.ModuleType('imp')
    _imp_shim.reload = importlib.reload
    sys.modules['imp'] = _imp_shim
%load_ext autoreload
%autoreload 2


In [ ]:
# == user configuration: edit before running ==
import os

# Path to the best checkpoint saved by JEPATrainer (keys: step, val_jepa, price_encoder, predictor, opt)
CHECKPOINT_PATH = '/content/drive/MyDrive/Colab Notebooks/JEPA-QUANT/checkpoints/nb03_best.pt'

# Directory containing per-ticker .parquet files (same as used for training / nb03c)
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/JEPA-QUANT/data/raw/stocks'

# HF TimesFM checkpoint (PyTorch). 2.0-500m: patch_length=32, hidden_size=1280, 50 layers.
# Our context L=64 = exactly 2 patches. The ForPrediction wrapper handles patching internally.
TIMESFM_CHECKPOINT = 'google/timesfm-2.0-500m-pytorch'

# Probe data caps. Collector stops early if loader exhausts, so overshooting is safe.
N_TRAIN_BATCHES = 100  # 100 * 256 = 25,600 samples nominal
N_VAL_BATCHES = 50     # 50  * 256 = 12,800 samples nominal

# WIDE ridge / logistic L2 sweep. Wider than Stage 1's (0.01..100) because Stage 1
# pinned best_alpha=100 (grid max) — under-regularized. TimesFM's D=1280 >> 256 wants
# more reg. We assert below that no representation pins best_alpha on a grid boundary;
# widen further if the assertion trips.
RIDGE_ALPHAS = (0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0)
LOG_ALPHAS = (0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0)

SEED = 42
DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print('device:', DEVICE)


In [ ]:
# == build config, load trained checkpoint, build untrained baseline ==
# Architecture matches nb03 / nb03c: d_model=768, n_layers=8, n_heads=12, freeze_backbone=True.
# (Only the z_price encoders use this config; TimesFM and raw-input ignore it.)
import torch
from jepa_quant.config import JEPAConfig, PriceEncoderConfig, DataConfig, TrainConfig
from jepa_quant.eval.diagnostics import load_checkpoint
from jepa_quant.training.trainer import build_components

cfg = JEPAConfig(
    price_encoder=PriceEncoderConfig(
        backend='transformer', n_features=6, context_length=64,
        d_model=768, n_heads=12, n_layers=8, latent_dim=256, freeze_backbone=True,
    ),
    data=DataConfig(
        data_dir=DATA_DIR, context_length=64, horizon=16, val_fraction=0.15, normalize=True,
    ),
    train=TrainConfig(batch_size=256, num_workers=2, device=DEVICE, seed=SEED),
)

# Trained: load the nb03 checkpoint (load_checkpoint puts the encoder on DEVICE + eval).
trained = load_checkpoint(CHECKPOINT_PATH, cfg, device=DEVICE)
print(f'trained loaded from {CHECKPOINT_PATH}')

# Untrained FLOOR: fresh random init, same architecture, no checkpoint.
torch.manual_seed(SEED)
untrained = build_components(cfg)
for mod in (untrained.price_encoder, untrained.target_encoder, untrained.predictor, untrained.regularizer):
    mod.to(DEVICE).eval()
print('untrained baseline built (random init, same arch)')


In [ ]:
# == build the FOUR encode_fns — all scored by the identical harness/seed/grid ==
# encode_fn signature: ctx_norm[B, L, F] (on DEVICE) -> features[B, D].
#   raw_input        : hand-crafted per-channel [mean,std,last,sum]  -> D=24  (no encoder)
#   untrained_zprice : random-init projection head + frozen random backbone -> D=256 (FLOOR)
#   trained_zprice   : nb03_best post-head latent                          -> D=256
#   timesfm          : RAW pretrained TimesFM pooled hidden states          -> D=1280 (CEILING)
# The z_price modules are already on DEVICE + eval (cell above); as encode_fns they are
# just the callable modules — the collector moves the input to DEVICE for them.
from jepa_quant.eval import raw_feature_encode_fn, timesfm_encode_fn

encoders = {
    'raw_input':        raw_feature_encode_fn(),
    'untrained_zprice': untrained.price_encoder,
    'trained_zprice':   trained.price_encoder,
    'timesfm':          timesfm_encode_fn(TIMESFM_CHECKPOINT, DEVICE, variant='close'),
}
ORDER = ['raw_input', 'untrained_zprice', 'trained_zprice', 'timesfm']
LABELS = {'raw_input': 'raw-input (baseline)', 'untrained_zprice': 'untrained z_price (FLOOR)',
          'trained_zprice': 'trained z_price', 'timesfm': 'TimesFM raw (CEILING)'}
print('encoders:', ORDER)


## Run the probe harness across all four encoders

Every representation goes through the SAME `probe_regression_features` /
`probe_direction_features` call (same `cfg`, `N_TRAIN_BATCHES`, `N_VAL_BATCHES`,
`SEED`, alpha grid) so the ceiling-vs-floor comparison is valid. The only thing
that varies is the `encode_fn`.

In [ ]:
# == regression: future_return and future_volatility, all four encoders ==
from jepa_quant.eval import probe_regression_features

reg_args = dict(n_train_batches=N_TRAIN_BATCHES, n_val_batches=N_VAL_BATCHES,
                ridge_alphas=RIDGE_ALPHAS, seed=SEED, device=DEVICE)

RET = {n: probe_regression_features(encoders[n], cfg, target_kind='future_return',
                                    source_label=n, **reg_args) for n in ORDER}
VOL = {n: probe_regression_features(encoders[n], cfg, target_kind='future_volatility',
                                    source_label=n, **reg_args) for n in ORDER}

def _reg_row(name, res):
    r = res[name]
    return (f"  {LABELS[name]:<26} val R^2 {r['val_r2']:+.4f}   "
            f"train R^2 {r['train_r2_at_best_alpha']:+.4f}   "
            f"gap {r['train_r2_at_best_alpha']-r['val_r2']:+.4f}   "
            f"best_a {r['best_alpha']:<8}  D={r['latent_dim']}")

print('=== future_return (ridge val R^2) ===')
for n in ORDER: print(_reg_row(n, RET))
print()
print('=== future_volatility (ridge val R^2) ===')
for n in ORDER: print(_reg_row(n, VOL))


In [ ]:
# == direction: sign(future_return), all four encoders ==
import math
from jepa_quant.eval import probe_direction_features

dir_args = dict(n_train_batches=N_TRAIN_BATCHES, n_val_batches=N_VAL_BATCHES,
                alphas=LOG_ALPHAS, seed=SEED, device=DEVICE)
DIR = {n: probe_direction_features(encoders[n], cfg, source_label=n, **dir_args) for n in ORDER}

n_val = DIR['trained_zprice']['n_val']
binom_se = math.sqrt(0.25 / n_val)
print('=== direction: sign(future_return) ===')
for n in ORDER:
    d = DIR[n]
    print(f"  {LABELS[n]:<26} acc {d['val_accuracy']:.4f}   AUROC {d['val_auroc']:.4f}   "
          f"best_a {d['best_alpha']:<8}")
print(f"  majority baseline: {DIR['trained_zprice']['majority_class_accuracy']:.4f}   "
      f"n_val={n_val}  binom_se={binom_se:.4f}  (signal threshold 2*se={2*binom_se:.4f})")


In [ ]:
# == 4x3 comparison table + comparability guards ==
# (1) Boundary-pinning guard: if best_alpha lands on the grid edge for ANY representation,
#     the sweep is too narrow and the reported R^2 is not the best-regularized one — widen.
# (2) Overfit watch: TimesFM D=1280 >> 256; large train-minus-val R^2 gap flags probe overfit.
amin, amax = min(RIDGE_ALPHAS), max(RIDGE_ALPHAS)
lmin, lmax = min(LOG_ALPHAS), max(LOG_ALPHAS)
pinned = []
for n in ORDER:
    for res, grid_lo, grid_hi, tag in [(RET[n], amin, amax, 'return'), (VOL[n], amin, amax, 'vol')]:
        if res['best_alpha'] in (grid_lo, grid_hi):
            pinned.append(f"{n}/{tag} best_alpha={res['best_alpha']}")
    if DIR[n]['best_alpha'] in (lmin, lmax):
        pinned.append(f"{n}/dir best_alpha={DIR[n]['best_alpha']}")

print('=== 4x3 ceiling-vs-floor table (val metrics) ===')
print(f"  {'representation':<26} {'ret R^2':>9} {'vol R^2':>9} {'dir acc':>9} {'dir AUROC':>10}")
for n in ORDER:
    print(f"  {LABELS[n]:<26} {RET[n]['val_r2']:>+9.4f} {VOL[n]['val_r2']:>+9.4f} "
          f"{DIR[n]['val_accuracy']:>9.4f} {DIR[n]['val_auroc']:>10.4f}")
print()
if pinned:
    print('  [WARN] best_alpha pinned on grid boundary — WIDEN the grid and re-run:')
    for p in pinned: print('        ', p)
else:
    print('  [OK] no representation pinned best_alpha on a grid boundary.')


In [ ]:
# == consolidated results_summary + automated verdict ==
C  = VOL['timesfm']['val_r2']           # ceiling: TimesFM vol val R^2
Fl = VOL['untrained_zprice']['val_r2']  # floor:   untrained random projection
Rw = VOL['raw_input']['val_r2']         # raw-input baseline

commit = (C > Fl + 0.03) and (C > Rw + 0.02)
hold = abs(C - Fl) <= 0.01
best_dir_acc = max(DIR[n]['val_accuracy'] for n in ORDER)
majority = DIR['trained_zprice']['majority_class_accuracy']
direction_dead = (best_dir_acc - majority) <= 2 * binom_se

if commit:
    verdict = 'COMMIT to TimesFM pivot: ceiling beats floor by >0.03 and raw by >0.02 on vol R^2.'
elif C < Fl - 0.01:
    verdict = 'REFUTE/HOLD: TimesFM is BELOW the random floor on vol R^2 -> a frozen-TimesFM swap does NOT help; do not pivot.'
elif hold:
    verdict = 'HOLD: TimesFM ~= random-projection floor on vol R^2; backbone swap alone insufficient.'
else:
    verdict = 'INTERMEDIATE: TimesFM above floor but below the commit margin; inspect per-target.'

results_summary = {
    'checkpoint': CHECKPOINT_PATH, 'timesfm_checkpoint': TIMESFM_CHECKPOINT,
    'n_train_batches': N_TRAIN_BATCHES, 'n_val_batches': N_VAL_BATCHES, 'seed': SEED,
    'ridge_alphas': list(RIDGE_ALPHAS), 'boundary_pinned': pinned,
    'future_return':     {n: {'val_r2': RET[n]['val_r2'], 'train_r2': RET[n]['train_r2_at_best_alpha'],
                              'best_alpha': RET[n]['best_alpha']} for n in ORDER},
    'future_volatility': {n: {'val_r2': VOL[n]['val_r2'], 'train_r2': VOL[n]['train_r2_at_best_alpha'],
                              'best_alpha': VOL[n]['best_alpha']} for n in ORDER},
    'direction':         {n: {'val_accuracy': DIR[n]['val_accuracy'], 'val_auroc': DIR[n]['val_auroc'],
                              'best_alpha': DIR[n]['best_alpha']} for n in ORDER},
    'decision_inputs': {'C_timesfm_vol_r2': C, 'Fl_untrained_vol_r2': Fl, 'Rw_raw_vol_r2': Rw,
                        'best_dir_acc': best_dir_acc, 'majority': majority, 'binom_se': binom_se},
    'verdict': verdict, 'commit': bool(commit), 'hold': bool(hold),
    'direction_near_majority': bool(direction_dead),
}
import json
print(json.dumps(results_summary, indent=2, default=float))
print()
print('VERDICT:', verdict)
if direction_dead:
    print('DIRECTION: best acc across all four ~= majority baseline -> drop direction as a primary target.')


## Summary and Decision

Fill in the **RESULT** and **DECISION + NEXT ACTION** fields in the header cell after running, quoting from `results_summary`.

* * *

### Decision tree

```
C  = TimesFM future_volatility val R^2   (CEILING)
Fl = untrained-random      vol val R^2   (FLOOR)
Rw = raw-input             vol val R^2   (baseline)

C > Fl + 0.03  AND  C > Rw + 0.02
  -> COMMIT to the TimesFM pivot. Signal exists in the pretrained representation that
     a random projection and raw inputs do not capture. The project's failure is the
     loop/head, not the backbone's information content. NEXT: build a JEPA loop on a
     FROZEN TimesFM backbone (no random-frozen backbone), benchmarked against THIS floor.

C ~= Fl  (within ~0.01)
  -> HOLD. TimesFM is no richer than a random projection for these targets. A backbone
     swap alone will not help. Reconsider the targets / horizon, or add conditioning
     (options, futures, text) before spending any training units.

best direction acc across ALL four ~= majority (~0.53)
  -> H=16 direction is near-unpredictable from price alone. Drop direction as a primary
     target; focus future_return / future_volatility, or add conditioning.

trained_zprice vol R^2  <  untrained_zprice vol R^2   (anti-informative, as in Stage 1)
  AND  TimesFM vol R^2  >  both
  -> Confirms the two-fault diagnosis: the trained head destroys signal (loop/head must
     change) AND TimesFM supplies signal the random backbone lacked (backbone swap helps).
     Both are needed; a pretrained backbone is necessary but not sufficient.
```

### Reporting in the project log

Quote at least: `C` / `Fl` / `Rw` (TimesFM vs floor vs raw on future_volatility val R²),
the train-vs-val R² gap for TimesFM (probe-overfit check at D=1280), the direction
acc vs majority across all four, and whether any `best_alpha` pinned on a grid boundary.
